In [ ]:
import eval_common as _ec
_ec.set_axis("mechanism")     # change to "lob" for the line-of-business axis IGNORE
AXIS = _ec.AXIS
print("AXIS =", AXIS, "| categories:", _ec.MECH_ORDER)

AXIS = mechanism | categories: ['Reliability', 'Bias & Fairness', 'Privacy, Confidentiality & Infringement', 'Security & Misuse', 'Autonomous Actions', 'Governance, Oversight & Explainability']


In [10]:
# %pip install catboost

In [11]:
from pathlib import Path
import numpy as np, pandas as pd
OUTPUT_DIR=Path("model_outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
DATE_COL="date"
import eval_common as _ec
MECH_ORDER = _ec.MECH_ORDER
df = _ec.load_incidents()
counts = _ec.build_counts(df)
print(counts.shape, counts.index[0], "->", counts.index[-1])

(127, 6) 2016-01 -> 2026-07


In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
evaluate = _ec.evaluate
mlog = _ec.mlog

In [ ]:
from catboost import CatBoostRegressor

def make_long_panel(C):
    """
    Convert the wide monthly count table into one row per
    month-mechanism combination.
    """
    rows = []
    monthly_totals = C.sum(axis=1)
    for month_index, month_period in enumerate(C.index):
        total = float(monthly_totals.loc[month_period])
        for category in MECH_ORDER:
            count_value = float(C.loc[month_period, category])
            rows.append({
                "month_period": month_period,
                "month_index": month_index,
                "year": month_period.year,
                "month_number": month_period.month,
                "category": category,
                "count": count_value,
                "share": count_value / total if total > 0 else 0.0,
                "total": total,
            })
    return pd.DataFrame(rows)

def add_temporal_features(panel, advanced=False):
    """
    Create lagged features using only previous months.
    Baseline version:
    - previous-month mechanism count;
    - previous-month mechanism share;
    - previous-month total incident count;
    - calendar and time features.
    """
    x = panel.sort_values(["category", "month_index"]).reset_index(drop=True).copy()
    lag_values = [1] if not advanced else [1, 2, 3, 6, 12]
    for lag in lag_values:
        x[f"count_lag_{lag}"] = x.groupby("category")["count"].shift(lag)
        x[f"share_lag_{lag}"] = x.groupby("category")["share"].shift(lag)
    monthly_total = (x[["month_index", "total"]].drop_duplicates(subset="month_index")
                     .sort_values("month_index").set_index("month_index")["total"])
    for lag in lag_values:
        lagged_total = monthly_total.shift(lag)
        x[f"total_lag_{lag}"] = x["month_index"].map(lagged_total)
    if advanced:
        for window in [3, 6, 12]:
            x[f"count_roll_mean_{window}"] = x.groupby("category")["count"].transform(
                lambda s: s.shift(1).rolling(window=window, min_periods=1).mean())
            x[f"share_roll_mean_{window}"] = x.groupby("category")["share"].transform(
                lambda s: s.shift(1).rolling(window=window, min_periods=1).mean())
            x[f"count_roll_std_{window}"] = x.groupby("category")["count"].transform(
                lambda s: s.shift(1).rolling(window=window, min_periods=2).std())
    return x

def feature_columns(advanced=False):
    baseline_features = ["category", "month_index", "month_number",
                         "count_lag_1", "share_lag_1", "total_lag_1"]
    if not advanced:
        return baseline_features
    advanced_features = []
    for lag in [2, 3, 6, 12]:
        advanced_features.extend([f"count_lag_{lag}", f"share_lag_{lag}", f"total_lag_{lag}"])
    for window in [3, 6, 12]:
        advanced_features.extend([f"count_roll_mean_{window}", f"share_roll_mean_{window}",
                                  f"count_roll_std_{window}"])
    return baseline_features + advanced_features

def forecast(train_counts, fold, advanced, params):
    history = train_counts.copy().astype(float)
    yr, half = fold
    start, end = _ec.fold_bounds(yr, half)
    future_months = pd.period_range(start=start, end=end, freq="M")
    training_panel = make_long_panel(history)
    training_features = add_temporal_features(training_panel, advanced=advanced)
    features = feature_columns(advanced=advanced)
    training_data = training_features.dropna(subset=features + ["count"]).reset_index(drop=True).copy()
    if training_data.empty:
        raise RuntimeError("No usable CatBoost training rows were created.")

    model = CatBoostRegressor(**params)
    model.fit(training_data[features], training_data["count"], cat_features=["category"])

    forecast_rows = []
    for future_month in future_months:
        placeholder = pd.DataFrame(np.zeros((1, len(MECH_ORDER)), dtype=float),
                                   index=pd.PeriodIndex([future_month], freq="M"),
                                   columns=MECH_ORDER)
        temporary_counts = pd.concat([history, placeholder])
        temporary_panel = make_long_panel(temporary_counts)
        temporary_features = add_temporal_features(temporary_panel, advanced=advanced)
        current_month = (temporary_features[temporary_features["month_period"] == future_month]
                         .set_index("category").reindex(MECH_ORDER).reset_index())
        missing_features = [c for c in features if c not in current_month.columns]
        if missing_features:
            raise KeyError(f"Missing forecast features: {missing_features}")
        predicted_counts = np.asarray(model.predict(current_month[features]), dtype=float)
        predicted_counts = np.clip(predicted_counts, 1e-8, None)
        predicted_shares = predicted_counts / predicted_counts.sum()
        history.loc[future_month, MECH_ORDER] = predicted_counts
        for category, predicted_count, predicted_share in zip(MECH_ORDER, predicted_counts, predicted_shares):
            forecast_rows.append({
                "month_period": future_month,
                "category": category,
                "predicted_count": float(predicted_count),
                "predicted_share": float(predicted_share),
            })
    return pd.DataFrame(forecast_rows), model

In [ ]:
PARAM_GRID = [
    {"iterations": 300, "depth": 3, "learning_rate": 0.03, "l2_leaf_reg": 5},
    {"iterations": 600, "depth": 3, "learning_rate": 0.03, "l2_leaf_reg": 10},
    {"iterations": 300, "depth": 4, "learning_rate": 0.03, "l2_leaf_reg": 10},
    {"iterations": 600, "depth": 4, "learning_rate": 0.03, "l2_leaf_reg": 20},
    {"iterations": 300, "depth": 4, "learning_rate": 0.05, "l2_leaf_reg": 10},
    {"iterations": 600, "depth": 5, "learning_rate": 0.03, "l2_leaf_reg": 20},
]

BASE_PARAMS = {
    "loss_function": "Poisson",
    "random_strength": 1.0,
    "random_seed": 42,
    "allow_writing_files": False,
    "verbose": False,
}

In [ ]:
def attach_actuals(forecast_output, actual_counts):
    out = forecast_output.copy()
    out["month_period"] = pd.PeriodIndex(out["month_period"], freq="M")
    actual_long = (actual_counts.rename_axis("month_period").reset_index()
                   .melt(id_vars="month_period", var_name="category", value_name="actual_count"))
    actual_long["actual_total"] = actual_long.groupby("month_period")["actual_count"].transform("sum")
    actual_long["actual_share"] = actual_long["actual_count"] / actual_long["actual_total"].clip(lower=1e-12)
    return out.merge(actual_long[["month_period", "category", "actual_count", "actual_share"]],
                     on=["month_period", "category"], how="inner")

def forecast_metrics(scored):
    mae = mean_absolute_error(scored["actual_share"], scored["predicted_share"])
    rmse = mean_squared_error(scored["actual_share"], scored["predicted_share"]) ** 0.5
    def monthly_log_score(g):
        p = np.clip(g["predicted_share"].to_numpy(float), 1e-12, 1.0)
        p /= p.sum()
        y = g["actual_count"].to_numpy(float)
        return np.sum(y * np.log(p)) / y.sum()
    log_score = scored.groupby("month_period").apply(monthly_log_score, include_groups=False).mean()
    return {"mae": mae, "rmse": rmse, "log_score": log_score}

def tune_catboost(train_counts, outer_fold):
    val_yr, val_half = _ec.prev_fold(*outer_fold)
    val_start, val_end = _ec.fold_bounds(val_yr, val_half)
    inner_train = train_counts[train_counts.index < val_start].copy()
    validation_counts = train_counts[(train_counts.index >= val_start) &
                                     (train_counts.index <= val_end)].copy()
    if inner_train.empty or validation_counts.empty:
        raise RuntimeError(f"Cannot create internal validation fold for {outer_fold}.")

    tuning_rows = []
    for candidate in PARAM_GRID:
        params = {**BASE_PARAMS, **candidate}
        forecast_output, _ = forecast(inner_train, (val_yr, val_half), advanced=False, params=params)
        scored = attach_actuals(forecast_output, validation_counts)
        metrics = forecast_metrics(scored)
        tuning_rows.append({
            "outer_fold": _ec.fold_label(*outer_fold),
            "validation_fold": _ec.fold_label(val_yr, val_half),
            **candidate,
            "validation_mae": metrics["mae"],
            "validation_rmse": metrics["rmse"],
            "validation_log_score": metrics["log_score"],
        })

    tuning_table = pd.DataFrame(tuning_rows)
    best = tuning_table.sort_values(
        ["validation_log_score", "validation_mae", "validation_rmse"],
        ascending=[False, True, True]).iloc[0]
    best_params = {
        **BASE_PARAMS,
        "iterations": int(best["iterations"]),
        "depth": int(best["depth"]),
        "learning_rate": float(best["learning_rate"]),
        "l2_leaf_reg": float(best["l2_leaf_reg"]),
    }
    return best_params, tuning_table

In [ ]:
rows = []
models = {}
tuning_results = []
for (yr, half) in _ec.TEST_FOLDS:
    fold = (yr, half)
    fold_str = _ec.fold_label(yr, half)
    start, end = _ec.fold_bounds(yr, half)
    train_counts = counts[counts.index < start].copy()
    test_counts = counts[(counts.index >= start) & (counts.index <= end)].copy()
    if train_counts.empty or test_counts.empty:
        print(f"Skipping {fold_str}: missing train/test data.")
        continue
    print(f"\nTuning within data available before {fold_str}; forecasting {fold_str}")
    try:
        best_params, fold_tuning = tune_catboost(train_counts, fold)
        tuning_results.append(fold_tuning)
        print("Selected:", {k: best_params[k] for k in ["iterations", "depth", "learning_rate", "l2_leaf_reg"]})
        forecast_output, fitted_model = forecast(train_counts, fold, advanced=False, params=best_params)
    except Exception as error:
        print(f"Forecast failed for {fold_str}: {type(error).__name__}: {error}")
        continue

    models[fold_str] = fitted_model
    scored = attach_actuals(forecast_output, test_counts)
    scored["model"] = "temporal_catboost_baseline_tuned"
    scored["fold"] = fold_str
    rows.append(scored[["model", "fold", "month_period", "category",
                        "actual_count", "actual_share", "predicted_share"]])

if not rows:
    raise RuntimeError("No tuned CatBoost predictions were created.")

pred = pd.concat(rows, ignore_index=True)
pred["month_period"] = pred["month_period"].astype(str)
tuning_results = pd.concat(tuning_results, ignore_index=True)


Tuning within data available before 2020-H1; forecasting 2020-H1
Selected: {'iterations': 300, 'depth': 3, 'learning_rate': 0.03, 'l2_leaf_reg': 5.0}

Tuning within data available before 2020-H2; forecasting 2020-H2
Selected: {'iterations': 300, 'depth': 3, 'learning_rate': 0.03, 'l2_leaf_reg': 5.0}

Tuning within data available before 2021-H1; forecasting 2021-H1
Selected: {'iterations': 300, 'depth': 3, 'learning_rate': 0.03, 'l2_leaf_reg': 5.0}

Tuning within data available before 2021-H2; forecasting 2021-H2
Selected: {'iterations': 300, 'depth': 4, 'learning_rate': 0.05, 'l2_leaf_reg': 10.0}

Tuning within data available before 2022-H1; forecasting 2022-H1
Selected: {'iterations': 300, 'depth': 3, 'learning_rate': 0.03, 'l2_leaf_reg': 5.0}

Tuning within data available before 2022-H2; forecasting 2022-H2
Selected: {'iterations': 300, 'depth': 4, 'learning_rate': 0.03, 'l2_leaf_reg': 10.0}

Tuning within data available before 2023-H1; forecasting 2023-H1
Selected: {'iterations': 3

In [ ]:
share_sums = pred.groupby(["fold", "month_period"])["predicted_share"].sum()
print("Predicted monthly share-sum range:", share_sums.min(), "to", share_sums.max())

overall, by_fold = evaluate(pred)
display(overall)
display(by_fold)

selected_params = (tuning_results.sort_values(
    ["outer_fold", "validation_log_score", "validation_mae"],
    ascending=[True, False, True]).groupby("outer_fold").head(1).reset_index(drop=True))
display(selected_params)

pred.to_csv(OUTPUT_DIR / f"predictions_catboost_baseline_tuned_{AXIS}.csv", index=False)
overall.to_csv(OUTPUT_DIR / f"metrics_catboost_baseline_tuned_overall_{AXIS}.csv", index=False)
by_fold.to_csv(OUTPUT_DIR / f"metrics_catboost_baseline_tuned_byfold_{AXIS}.csv", index=False)
tuning_results.to_csv(OUTPUT_DIR / f"catboost_baseline_tuning_results_{AXIS}.csv", index=False)

Predicted monthly share-sum range: 0.9999999999999998 to 1.0000000000000002
Overall tuned CatBoost performance


,model,mae,rmse,mean_log_score_per_incident
0,temporal_catboost_baseline_tuned,0.092805,0.124828,-1.64466


Performance by fold


,model,fold,mae,rmse
0,temporal_catboost_baseline_tuned,2020-H1,0.164638,0.208802
1,temporal_catboost_baseline_tuned,2020-H2,0.105280,0.132334
2,temporal_catboost_baseline_tuned,2021-H1,0.110827,0.132377
3,temporal_catboost_baseline_tuned,2021-H2,0.120100,0.137044
4,temporal_catboost_baseline_tuned,2022-H1,0.086233,0.103501
5,temporal_catboost_baseline_tuned,2022-H2,0.101573,0.128088
6,temporal_catboost_baseline_tuned,2023-H1,0.104963,0.150037
7,temporal_catboost_baseline_tuned,2023-H2,0.110220,0.144587
8,temporal_catboost_baseline_tuned,2024-H1,0.067287,0.091227
9,temporal_catboost_baseline_tuned,2024-H2,0.080513,0.114274


Hyperparameter validation results


,outer_fold,validation_fold,iterations,depth,learning_rate,l2_leaf_reg,validation_mae,validation_rmse,validation_log_score
0,2020-H1,2019-H2,300,3,0.03,5,0.181205,0.245146,-1.559241
2,2020-H1,2019-H2,300,4,0.03,10,0.184065,0.245886,-1.573818
3,2020-H1,2019-H2,600,4,0.03,20,0.189743,0.250006,-1.629168
4,2020-H1,2019-H2,300,4,0.05,10,0.191040,0.253911,-1.644736
5,2020-H1,2019-H2,600,5,0.03,20,0.192634,0.255204,-1.657507
...,...,...,...,...,...,...,...,...,...
73,2026-H1,2025-H2,600,3,0.03,10,0.044299,0.057650,-1.399053
74,2026-H1,2025-H2,300,4,0.03,10,0.045337,0.059761,-1.400260
75,2026-H1,2025-H2,600,4,0.03,20,0.048628,0.065076,-1.406135
72,2026-H1,2025-H2,300,3,0.03,5,0.048765,0.063731,-1.409487


Selected parameters by fold


,outer_fold,validation_fold,iterations,depth,learning_rate,l2_leaf_reg,validation_mae,validation_rmse,validation_log_score
0,2020-H1,2019-H2,300,3,0.03,5,0.181205,0.245146,-1.559241
1,2020-H2,2020-H1,300,3,0.03,5,0.164638,0.208802,-1.982558
2,2021-H1,2020-H2,300,3,0.03,5,0.105280,0.132334,-1.713696
3,2021-H2,2021-H1,300,4,0.05,10,0.105424,0.127903,-1.795067
4,2022-H1,2021-H2,300,3,0.03,5,0.115803,0.133419,-1.771620
5,2022-H2,2022-H1,300,4,0.03,10,0.084631,0.101099,-1.722630
6,2023-H1,2022-H2,300,4,0.03,10,0.101573,0.128088,-1.776596
7,2023-H2,2023-H1,600,4,0.03,20,0.097854,0.141514,-1.704373
8,2024-H1,2023-H2,300,4,0.05,10,0.112118,0.144805,-1.693093
9,2024-H2,2024-H1,300,3,0.03,5,0.070016,0.093943,-1.453555
